# Dam removal effects by distance downstream

**Question:** after a dam is removed, does water quality change downstream, how far does the change reach, and is it bigger for bigger reservoirs?

**How to read every result**
- **Effect** = change after removal (log10 units for skewed measures: 0.1 = about +26%).
- **95% range** = plausible values for the effect. If it includes 0, we can't tell whether anything changed.
- **Verdict:** <span style="color:#c0392b">**detected**</span> (range excludes 0) · <span style="color:#27ae60">**no meaningful change**</span> (whole range inside the "too small to matter" threshold) · <span style="color:#7f8c8d">**inconclusive**</span> (can't tell).
- **Effect ÷ threshold:** most charts put every measure on one scale by dividing by its "too small to matter" threshold, so **±1 = the threshold** (e.g. 1 = 0.5 °C for temperature, a 25% change for turbidity).
- Results are associations, not proof of cause. Fewer than 15 dams = *exploratory*.

**Glossary**
- **BACI** (Before-After-Control-Impact): downstream change minus the change at upstream "control" stations of the same dam. Removes regional trends (weather, climate) that affect both. Strong evidence.
- **Before/after:** downstream change only. More dams, but can't separate the removal from regional trends. Supporting evidence.
- **Mixed-effects model:** a regression that lets each dam and each station have its own baseline (*random effects*), so many measurements from one dam aren't treated as independent evidence.
- **Drainage-area ratio:** station's upstream area ÷ dam's. About 1 = the station's water comes mostly through the dam; 100 = the dam supplies about 1%.
- **Smallest detectable effect (MDE):** how big a real change would need to be for the model to detect it 80% of the time.

Run all cells; nothing is refit (`fit_distance_models.slurm` and `fit_stage4.slurm` produce the inputs).

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Find the project folder and load the report tables written by the cluster jobs.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "etl").is_dir() and (p / "config").is_dir())
sys.path.insert(0, str(ROOT))
from analysis import mixed_model as mm
from analysis import stage4_report as rep

CASE_STUDY = "all_removals"
MODEL_DIR = Path(os.environ.get("DAM_REMOVAL_MODEL_DIR", mm.output_dir_for(CASE_STUDY, ROOT)))
REPORT_DIR, INFLUENCE_DIR = MODEL_DIR / "stage4" / "report", MODEL_DIR / "stage4" / "influence"
if not (REPORT_DIR / "model_summary.csv").exists():      # rebuild tables if missing (seconds)
    rep.build(MODEL_DIR, REPORT_DIR, figures=False)

def read(name):
    path = REPORT_DIR / f"{name}.csv"
    return pd.read_csv(path) if path.exists() and path.stat().st_size > 1 else pd.DataFrame()

summary, by_distance, curve = read("model_summary"), read("effects_by_distance"), read("effects_curve")
sensitivity, comparison, influence = read("sensitivity_table"), read("distance_comparison"), read("influence_summary")
thresholds, threshold_sensitivity = read("sesoi_thresholds"), read("sesoi_sensitivity")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.3g}")

# Discharge rows are a sampling check (section 9), kept out of the main charts.
is_check = (summary["purpose"].eq("sampling-conditions check") if "purpose" in summary
            else pd.Series(False, index=summary.index))
ok = summary["error"].fillna("").eq("")
primary = summary[(summary["run"] == rep.PRIMARY[0]) & (summary["re"] == rep.PRIMARY[1]) & ~is_check & ok]
check = summary[(summary["run"] == rep.PRIMARY[0]) & (summary["re"] == rep.PRIMARY[1]) & is_check]

COLORS = {"detected": "#c0392b", "no meaningful change": "#27ae60", "inconclusive": "#7f8c8d"}
DESIGN = {"baci_distance": "BACI", "before_after_distance": "before/after"}

def forest(ax, frame, est, lo, hi, label, color_col=None, title="", xlabel="effect ÷ threshold"):
    """One row per estimate: dot + 95% range, colored by verdict; grey band = ±1 threshold."""
    frame = frame.reset_index(drop=True)
    y = np.arange(len(frame))[::-1]
    colors = (frame[color_col].map(COLORS).fillna("#34495e") if color_col
              else pd.Series("#34495e", index=frame.index))
    ax.axvspan(-1, 1, color="0.9", lw=0); ax.axvline(0, color="black", lw=0.8)
    for yi, (_, r), color in zip(y, frame.iterrows(), colors):
        ax.plot([r[lo], r[hi]], [yi, yi], color=color, lw=1.5)       # 95% range
        ax.plot(r[est], yi, "o", color=color)                         # estimate
    ax.set_yticks(y, frame[label], fontsize=8); ax.set_title(title, fontsize=10); ax.set_xlabel(xlabel)

def per_threshold(frame, cols):
    """Divide the given columns by each row's threshold (±1 = too small to matter)."""
    frame = frame.copy()
    scale = frame["series"].map(rep.SESOI)
    for c in cols:
        frame[c] = frame[c] / scale
    return frame

print(f"Model outputs: {MODEL_DIR}")

## The models

One model per water-quality measure, fit on daily values (one per station per day). Each removal is compared 5 years before vs 5 years after.

**BACI distance model** (main analysis):

$$y = \beta_0 + \beta_1\,\text{post} + \beta_2\,\text{down} + \beta_3\,\text{post}{\times}\text{down} + \beta_4\,\text{down}{\times}d + \beta_5\,\text{post}{\times}\text{down}{\times}d + \beta_6\,\text{down}{\times}r + \beta_7\,\text{post}{\times}\text{down}{\times}r + \text{month} + \text{weather} + u_{\text{dam}} + v_{\text{station}} + \varepsilon$$

- *post* = after removal (0/1); *down* = downstream station (0/1; upstream controls = 0)
- *d* = ln(1 + km downstream) − ln(6), so *d* = 0 at **5 km**; 0 for upstream stations
- *r* = log10(drainage-area ratio) − its median: how much other water has joined (dilution)
- *month* = season; *weather* = air temperature and last month's rain
- *u*<sub>dam</sub>, *v*<sub>station</sub> = each dam's and station's own baseline. Where the data support it, each dam also gets its own effect (random slopes); see section 4.

**Effect at distance d** = β₃ + β₅·d. So **β₃ = the effect at 5 km** and **β₅ = how the effect changes with distance** (negative β₅ with a positive effect = it fades downstream).

**Before/after distance model** (supporting): the same without upstream controls:

$$y = \beta_0 + \beta_1\,\text{post} + \beta_2\,d + \beta_3\,\text{post}{\times}d + \beta_4\,r + \beta_5\,\text{post}{\times}r + \text{month} + \text{weather} + u_{\text{dam}} + v_{\text{station}} + \varepsilon \qquad \text{effect} = \beta_1 + \beta_3 d$$

## Where are the dams?

Every dam removal used in the main distance models. **Red** = at least one measure had upstream control stations (BACI, strong evidence); **blue** = downstream stations only (before/after). Bigger circles = used for more measures. **Use the layer box (top right) to show the dams behind one measure**, e.g. only the dissolved-oxygen dams. Click a dam for its name, removal year, and the measures and stations it contributed.

In [ ]:
import folium
from etl.common.config import load_config
from etl.common.warehouse import warehouse_dir

used_path = MODEL_DIR / "stage3" / "primary__auto" / "dams_used.csv"
try:   # the warehouse dam table has names and coordinates
    WAREHOUSE = Path(os.environ.get("DAM_REMOVAL_WAREHOUSE_DIR") or warehouse_dir(load_config(CASE_STUDY)))
except Exception:
    WAREHOUSE = None
if not used_path.exists() or WAREHOUSE is None or not (WAREHOUSE / "dam_removal.parquet").exists():
    print("Map needs stage3/primary__auto/dams_used.csv (rerun: RUNS=primary sbatch "
          "data/slurm/fit_distance_models.slurm) and the warehouse dam table.")
else:
    used = pd.read_csv(used_path, dtype={"AR_ID": str})
    used = used[used["spec"].isin(DESIGN) & ~used["series"].isin(rep.DIAGNOSTIC_SERIES)]
    dams = pd.read_parquet(WAREHOUSE / "dam_removal.parquet",
                           columns=["AR_ID", "Dam_Name", "River", "State", "Year_Removed", "Latitude", "Longitude"])
    dams = dams.assign(AR_ID=dams["AR_ID"].astype(str)).set_index("AR_ID")

    # Per dam: the measures it was used for, and whether any used upstream controls.
    per_dam = used.groupby("AR_ID").agg(n_measures=("series", "nunique"),
                                        baci=("design", lambda d: d.eq("baci").any()))
    per_dam = per_dam.join(dams, how="inner").dropna(subset=["Latitude", "Longitude"])
    RED, BLUE = "#c0392b", "#2980b9"

    def popup(aid, row, series=None):
        rows = used[used["AR_ID"].eq(aid) & (used["series"].eq(series) if series else True)]
        table = "".join(f"<tr><td>{s}</td><td>{'BACI' if d == 'baci' else 'before/after'}</td>"
                        f"<td>{t} down, {c} up</td></tr>"
                        for s, d, t, c in rows[["series", "design", "treated_stations", "control_stations"]]
                        .itertuples(index=False))
        return folium.Popup(f"<b>{row.Dam_Name}</b> ({aid})<br>{row.River}, {row.State}; removed "
                            f"{row.Year_Removed}<table>{table}</table>", max_width=360)

    m = folium.Map(location=[39.5, -96], zoom_start=4, tiles="CartoDB positron")
    everything = folium.FeatureGroup(name=f"All dams ({len(per_dam)})", show=True)
    for aid, row in per_dam.iterrows():
        color = RED if row.baci else BLUE
        folium.CircleMarker([row.Latitude, row.Longitude], radius=3 + 0.7 * row.n_measures, color=color,
                            fill=True, fill_opacity=0.7, weight=1, popup=popup(aid, row),
                            tooltip=f"{row.Dam_Name}: {row.n_measures} measures").add_to(everything)
    everything.add_to(m)
    # One layer per measure (off by default): the dams behind that measure's models.
    for series, g in used.groupby("series"):
        baci = g.groupby("AR_ID")["design"].apply(lambda d: d.eq("baci").any())
        layer = folium.FeatureGroup(name=f"{series} ({g['AR_ID'].nunique()} dams, {int(baci.sum())} BACI)", show=False)
        for aid, is_baci in baci.items():
            if aid in per_dam.index:
                row = per_dam.loc[aid]
                folium.CircleMarker([row.Latitude, row.Longitude], radius=5, color=RED if is_baci else BLUE,
                                    fill=True, fill_opacity=0.8, weight=1, popup=popup(aid, row, series),
                                    tooltip=row.Dam_Name).add_to(layer)
        layer.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.get_root().html.add_child(folium.Element(
        f'<div style="position:fixed;bottom:25px;left:25px;z-index:9999;background:white;padding:6px 10px;'
        f'border:1px solid #999;font-size:12px"><span style="color:{RED}">&#9679;</span> has upstream controls (BACI)'
        f'<br><span style="color:{BLUE}">&#9679;</span> downstream only (before/after)</div>'))
    display(m)

## 1. Main results

**Top:** effect at 5 km. **Bottom:** change with distance (left of 0 = a positive effect shrinks downstream). Everything is ÷ threshold, so the grey band (±1) is "too small to matter". Labels show the number of dams.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for col, (spec, design) in enumerate(DESIGN.items()):
    f = primary[primary["spec"] == spec].copy()
    f["label"] = f["series"] + " (" + f["dams"].astype(int).astype(str) + ")"
    f = per_threshold(f, ["effect_at_reference", "effect_ci_low", "effect_ci_high",
                          "modifier_slope", "modifier_ci_low", "modifier_ci_high"])
    forest(axes[0, col], f, "effect_at_reference", "effect_ci_low", "effect_ci_high", "label",
           "effect_class", title=f"{design}: effect at 5 km")
    forest(axes[1, col], f, "modifier_slope", "modifier_ci_low", "modifier_ci_high", "label",
           "slope_class", title=f"{design}: change with distance", xlabel="slope ÷ threshold")
fig.tight_layout(); plt.show()
skipped = summary[(summary["run"] == "primary") & (summary["re"] == "auto") & ~ok & ~is_check]
if len(skipped):
    print("Not fit (too few dams):", ", ".join(sorted(set(skipped["series"] + " " + skipped["spec"].map(DESIGN).fillna("")))))

## 2. How the effect changes with distance

**Line** = best estimate; **shaded band** = 95% range; **grey band** = too small to matter; **black line** = no change. Where the shaded band covers 0 we can't tell. A line moving toward 0 going right = the effect fades downstream. **Orange dashed** = a simpler model assuming every dam had the same effect (a robustness check: close to blue = the choice doesn't matter). Distance axis is logarithmic.

In [ ]:
for series in dict.fromkeys(curve["series"]):
    if series in rep.DIAGNOSTIC_SERIES:          # shown in section 9
        continue
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    rep.plot_distance_response(curve, by_distance, series, axes, rep.SESOI.get(series))
    fig.tight_layout(); plt.show()

**Verdict at each distance** (color) with the effect (number, in the measure's units). Blank = no stations that far downstream.

In [ ]:
pts = by_distance[by_distance["primary"] & by_distance["distance_km"].notna()
                 & by_distance["spec"].isin(DESIGN) & ~by_distance["series"].isin(rep.DIAGNOSTIC_SERIES)]
code_of = {"detected": 2, "no meaningful change": 1, "inconclusive": 0}
from matplotlib.colors import ListedColormap
cmap = ListedColormap([COLORS["inconclusive"], COLORS["no meaningful change"], COLORS["detected"]])
fig, axes = plt.subplots(1, 2, figsize=(12, 0.45 * max(pts["series"].nunique(), 2) + 1.5))
for ax, (spec, design) in zip(axes, DESIGN.items()):
    g = pts[pts["spec"] == spec]
    if g.empty:
        ax.set_visible(False); continue
    grid = g.pivot_table(index="series", columns="distance_km", values="effect_class", aggfunc="first")
    vals = g.pivot_table(index="series", columns="distance_km", values="estimate", aggfunc="first")
    ax.imshow(grid.apply(lambda c: c.map(code_of)).to_numpy(dtype=float), cmap=cmap, vmin=0, vmax=2, aspect="auto")
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            if pd.notna(vals.iat[i, j]):
                ax.text(j, i, f"{vals.iat[i, j]:+.2f}", ha="center", va="center", color="white", fontsize=8)
    ax.set_xticks(range(grid.shape[1]), [f"{int(c)} km" for c in grid.columns])
    ax.set_yticks(range(grid.shape[0]), grid.index, fontsize=8); ax.set_title(design)
fig.tight_layout(); plt.show()

## 3. Do the results depend on our choices?

The effect at 5 km re-estimated six ways: the main model, a simpler model, only stations where the dam supplies at least 1% / 10% / 50% of the river's water, and with the 12 suspect dams kept in. **Dots that line up = a robust result.** The table says whether every run agrees on direction and on whether a change was detected.

In [ ]:
KM = 5   # change to 1, 10 or 20 to check another distance
RUNS = {"primary": "main", "primary (intercepts)": "simpler model", "dar_le_100": "dam >=1% of flow",
        "dar_le_10": "dam >=10% of flow", "dar_le_2": "dam >=50% of flow", "keep_suspects": "suspect dams kept"}
t = sensitivity[sensitivity["distance_km"] == KM].copy()
if t.empty:
    print(f"No estimates at {KM} km")
else:
    t["run"] = t["model_run"].map(RUNS).fillna(t["model_run"])
    t = per_threshold(t, ["estimate", "ci_low", "ci_high"])
    groups = list(t.groupby(["design", "series"]))
    ncol = 4
    fig, axes = plt.subplots(int(np.ceil(len(groups) / ncol)), ncol, figsize=(15, 2.4 * np.ceil(len(groups) / ncol)),
                             squeeze=False)
    for ax, ((design, series), g) in zip(axes.flat, groups):
        g = g.set_index("run").reindex([r for r in RUNS.values() if r in set(g["run"])]).reset_index()
        forest(ax, g, "estimate", "ci_low", "ci_high", "run", "effect_class", title=f"{series} ({design})")
        ax.set_xlabel("")
    for ax in list(axes.flat)[len(groups):]:
        ax.set_visible(False)
    fig.suptitle(f"Effect at {KM} km ÷ threshold, across runs", fontsize=11); fig.tight_layout(); plt.show()
    t["detected"] = t["effect_class"].eq("detected")
    robust = t.groupby(["series", "design"]).apply(
        lambda g: "yes" if np.sign(g["estimate"]).nunique() == 1 and g["detected"].nunique() == 1 else "no")
    display(robust.rename("same answer in every run?").unstack("design"))

## 4. One effect for every dam, or does each dam differ?

The main model gives each dam its own effect (*random slopes*) only where the data support it, judged by a **likelihood-ratio test (LRT)**: does letting dams differ improve the fit more than chance would? Because a variance can't be negative, the usual chi-square p-value is too cautious here, so borderline cases use a simulation (*parametric bootstrap*) instead.

Bars: how much wider the main model's 95% range is than one assuming a single shared effect. **Above 1 = dams genuinely differ**, and the simpler model would overstate certainty. Temperature (BACI) found no difference, so it uses the simpler model (ratio = 1).

In [ ]:
auto = summary[(summary["run"] == "primary") & (summary["re"] == "auto") & ~is_check & ok]
flat = summary[(summary["run"] == "primary") & (summary["re"] == "intercepts") & ~is_check & ok]
if flat.empty:
    print("No intercept-only run to compare (RE=intercepts).")
else:
    w = auto.merge(flat, on=["series", "spec"], suffixes=("", "_flat"))
    w["ratio"] = (w["effect_ci_high"] - w["effect_ci_low"]) / (w["effect_ci_high_flat"] - w["effect_ci_low_flat"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    for ax, (spec, design) in zip(axes, DESIGN.items()):
        g = w[w["spec"] == spec].sort_values("ratio")
        ax.barh(g["series"], g["ratio"], color=np.where(g["re_choice"].str.startswith("slopes"), "#2980b9", "#bdc3c7"))
        ax.axvline(1, color="black", lw=0.8); ax.set_title(f"{design} (blue = each dam its own effect)", fontsize=10)
        ax.set_xlabel("width of 95% range: main ÷ single shared effect"); ax.tick_params(axis="y", labelsize=8)
    fig.tight_layout(); plt.show()

## 5. Is any result driven by a single dam?

**Bars:** the change in the 5 km effect when each dam is left out, in standard errors (SE = the typical uncertainty of the estimate). **Red bars beyond ±1 = influential dams.**

**Dots and lines below:** three 95% ranges per measure: the model's, a **bootstrap** (re-fitting 200 times on dams drawn at random with replacement), and the spread when leaving one dam out at a time. Similar ranges = trustworthy uncertainty; if they differ, trust the widest.

In [ ]:
lodo_path = INFLUENCE_DIR / "lodo.csv"
if lodo_path.exists():
    lodo = pd.read_csv(lodo_path)
    names = list(dict.fromkeys(lodo["series"]))
    fig, axes = plt.subplots(len(names), 1, figsize=(12, 1.9 * len(names)), squeeze=False)
    for ax, s in zip(axes[:, 0], names):
        g = lodo[lodo["series"] == s].sort_values("delta_effect_at_reference_in_se")
        ax.bar(g["dam_left_out"].astype(str), g["delta_effect_at_reference_in_se"],
               color=np.where(g["influential"].fillna(False).astype(bool), "#c0392b", "#2980b9"))
        for y in (-1, 1):
            ax.axhline(y, ls=":", color="grey")
        ax.set_title(f"{s}: change when each dam is left out (SEs)", fontsize=9)
        ax.tick_params(axis="x", labelrotation=90, labelsize=6)
    fig.tight_layout(); plt.show()
if len(influence) and "effect_at_reference_boot_low" in influence:
    rows = []
    for _, r in influence.iterrows():
        for name, lo, hi in (("model", "full_effect_ci_low", "full_effect_ci_high"),
                             ("bootstrap", "effect_at_reference_boot_low", "effect_at_reference_boot_high"),
                             ("leave one out", "lodo_effect_min", "lodo_effect_max")):
            rows.append({"series": r["series"], "kind": name, "est": r["full_effect_at_reference"], "lo": r[lo], "hi": r[hi]})
    iv = per_threshold(pd.DataFrame(rows), ["est", "lo", "hi"])
    iv["label"] = iv["series"] + ": " + iv["kind"]
    fig, ax = plt.subplots(figsize=(8, 0.35 * len(iv) + 1))
    forest(ax, iv, "est", "lo", "hi", "label", title="BACI effect at 5 km: three ways to measure uncertainty")
    fig.tight_layout(); plt.show()

## 6. Distance in km, or in reservoir lengths?

The advisor suggested bigger reservoirs may affect water further downstream. We compare two ways of measuring distance on **exactly the same dams and data**: kilometres (*d* above) vs reservoir lengths, *s* = ln(1 + km ÷ reservoir length) − ln(2), which replaces *d* in the same formula.

**AIC** (Akaike Information Criterion) scores how well a model fits the data, with a penalty for complexity; **lower is better**. Bars show **ΔAIC = AIC(reservoir lengths) − AIC(km)**: left of 0 (blue) = reservoir lengths fit better; right (orange) = km fits better. Within ±2 (grey band) = no real difference; beyond ±10 = strong preference.

In [ ]:
if comparison.empty:
    print("No comparison run yet.")
else:
    c = comparison.copy()
    c["label"] = c["series"] + " (" + c["design"].replace({"baci": "BACI", "before_after": "before/after"}) + ", " + c["dams"].astype(int).astype(str) + " dams)"
    c = c.sort_values("delta_aic_scaled_minus_raw")
    fig, ax = plt.subplots(figsize=(9, 0.4 * len(c) + 1))
    ax.axvspan(-2, 2, color="0.9", lw=0); ax.axvline(0, color="black", lw=0.8)
    ax.barh(c["label"], c["delta_aic_scaled_minus_raw"],
            color=np.where(c["delta_aic_scaled_minus_raw"] < 0, "#2980b9", "#e67e22"))
    ax.set_xlabel("ΔAIC  (← reservoir lengths fit better | km fit better →)"); ax.tick_params(axis="y", labelsize=8)
    fig.tight_layout(); plt.show()
    print("All comparisons on identical data:", bool(c["same_sample"].all() and c["same_covariates"].all()))

## 7. Which side moved? (pH, with temperature and oxygen for comparison)

BACI effect = downstream change − upstream change, so a positive BACI result can come from downstream rising **or** upstream falling. **Left:** values over time relative to before removal (orange = upstream controls, blue = downstream). **Right:** one dot per dam; above the dashed line = downstream rose more than upstream. For pH, check which line moved.

In [ ]:
for series in ["ph", "water_temp", "dissolved_oxygen"]:
    tp, cp = INFLUENCE_DIR / f"trajectory_{series}.csv", INFLUENCE_DIR / f"components_{series}.csv"
    if not tp.exists():
        continue
    traj, comp = pd.read_csv(tp), pd.read_csv(cp)
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
    for role, color, label in (("control", "#e67e22", "upstream controls"), ("treated", "#2980b9", "downstream")):
        g = traj[traj["role"] == role].sort_values("years_from_removal")
        axes[0].errorbar(g["years_from_removal"], g["mean"], yerr=1.96 * g["se"], color=color,
                         marker="o", capsize=2, label=label)
    axes[0].axvline(0, color="black", lw=0.8); axes[0].axhline(0, color="grey", lw=0.6)
    axes[0].set(xlabel="years from removal", ylabel="change vs before removal", title=series); axes[0].legend()
    axes[1].scatter(comp["change_upstream"], comp["change_downstream"], color="#2980b9")
    lim = np.nanmax(np.abs(comp[["change_upstream", "change_downstream"]].to_numpy())) * 1.1
    axes[1].plot([-lim, lim], [-lim, lim], color="grey", ls="--", lw=0.8)
    axes[1].axhline(0, color="grey", lw=0.5); axes[1].axvline(0, color="grey", lw=0.5)
    axes[1].set(xlabel="upstream change", ylabel="downstream change", title=f"{series}: one dot per dam")
    fig.tight_layout(); plt.show()

## 8. Bigger reservoirs, bigger effects? (exploratory)

The size model replaces distance with reservoir size, holding river size fixed:

$$y = \ldots + \beta_3\,\text{post}{\times}\text{down} + \beta_5\,\text{post}{\times}\text{down}{\times}z_{\text{size}} + \beta_7\,\text{post}{\times}\text{down}{\times}z_{\text{river}} + \ldots$$

*z*<sub>size</sub> = standardized log reservoir size (an index from dam height, slope and river size, or the mapped reservoir area); *z*<sub>river</sub> = standardized log drainage area. **β₅ = how much the effect changes per standard deviation of reservoir size.** Right of 0 = bigger effect for bigger reservoirs. Few dams have the data, so expect wide ranges. `time_and_modifier_results.ipynb` (section 8) tests reservoir size alongside dam height, river size, land cover and distance.

In [ ]:
size = summary[summary["run"].isin(["size_proxy", "size_polygon"]) & summary["re"].eq("auto") & ~is_check & ok].copy()
if size.empty:
    print("No size models fit.")
else:
    size["label"] = (size["series"] + " · " + size["spec"].str.replace("_size", "").str.replace("_", "/")
                     + " · " + size["run"].str.replace("size_", "") + " (" + size["dams"].astype(int).astype(str) + ")")
    size = per_threshold(size, ["modifier_slope", "modifier_ci_low", "modifier_ci_high"])
    fig, ax = plt.subplots(figsize=(9, 0.35 * len(size) + 1))
    forest(ax, size, "modifier_slope", "modifier_ci_low", "modifier_ci_high", "label", "slope_class",
           title="Change in effect per SD of reservoir size", xlabel="size slope ÷ threshold")
    fig.tight_layout(); plt.show()

## 9. Check: were samples taken at similar flows before and after?

Discharge (stream flow measured with each sample) is modeled like the others, but as a **check, not a finding**: small dam removals rarely change flow. **Inconclusive or near 0 = good** (similar flows before and after). **Detected = caution:** flow-sensitive results (turbidity, sediment, conductance, oxygen) may partly reflect different flow conditions.

In [ ]:
cols = ["spec", "dams", "effect_at_reference", "effect_ci_low", "effect_ci_high", "effect_class", "error"]
display(check[[c for c in cols if c in check]].rename(columns={"effect_at_reference": "effect at 5 km (log10 ft3/s)",
        "effect_ci_low": "low", "effect_ci_high": "high", "effect_class": "verdict"}) if len(check)
        else pd.DataFrame({"note": ["no discharge fits (rerun fit_distance_models.slurm)"]}))

## 10. How much do the verdicts depend on the "too small to matter" thresholds?

The thresholds are **our own choices, not environmental standards** (table). The chart re-labels every main estimate with thresholds half as big up to twice as big. **Bars that change color from left to right are verdicts that depend on the threshold**, so quote the numbers, not just the labels.

**Basis and caveats** (`docs/practical_significance_thresholds.md`, added 2026-11-09 after the results were seen; no value changed): the thresholds were informed by aquatic-life research, water-quality criteria and field-instrument accuracy, but published criteria are almost all *absolute* levels (e.g. DO < 5 mg/L, ecoregional phosphorus), not sizes of change, so no source fixes these values. They are single national values: the importance of a change really depends on baseline level, region and exposure duration. Weakest supported: **pH** (±0.1 equals single-reading sensor accuracy; kept, because both pH verdicts are *detected* and so don't depend on it, and moving it after seeing the results would be a post-hoc choice; read pH at 2× = ±0.2 here too), **TDS**, **orthophosphate** and **TSS** (not comparable with SSC).

In [ ]:
display(thresholds[["series", "threshold"]] if len(thresholds) else thresholds)
if len(threshold_sensitivity):
    cols = [c for c in threshold_sensitivity if c.startswith("class_at_")]
    counts = pd.DataFrame({c.replace("class_at_", ""): threshold_sensitivity[c].value_counts() for c in cols}).T.fillna(0)
    counts = counts[[k for k in COLORS if k in counts]]
    ax = counts.plot.bar(stacked=True, color=[COLORS[k] for k in counts], figsize=(7, 3.5), rot=0)
    ax.set_xlabel("threshold multiplier"); ax.set_ylabel("number of estimates")
    ax.set_title(f"{int(threshold_sensitivity['class_changes'].sum())} of {len(threshold_sensitivity)} verdicts change with the threshold")
    plt.tight_layout(); plt.show()

## 11. Sensitivity: a looser eligibility rule (3 samples, 1 year)

The main results use stations with **at least 3 samples in at least 2 different years** before and after removal. As a pre-specified check (`docs/eligibility_sensitivity_prespec.md`, written before these models were fit), the same models were refit with **3 samples in at least 1 year** (*3 / 1*), and with a stricter version that also needs **2 different months** when a side has only one year (*3 / 1, ≥2 months*). The main results stay the main results; this section asks whether they hold, and whether the extra dams help.

**Labels** (fixed in advance), sensitivity vs main: **robust** = same verdict and direction · **sharper** = main inconclusive, sensitivity decided · **weaker** = main decided, sensitivity inconclusive · **conflicting** = direction flips while a range excludes 0, or detected ↔ no meaningful change · **direction unstable** = flips, both inconclusive. A result found only under 3 / 1 is suggestive, never a headline.

Fewer than 10 dams on either side: shown, not compared. **Clarified 2026-11-08, after these results were seen:** both *no meaningful change* (both 95% ranges inside the threshold) counts as **robust** even if the signs differ (`negligible_sign_flip` marks those rows); estimates and thresholds are unchanged.

In [ ]:
from analysis import sensitivity_compare as sc

VARIANTS = {"3 / 1": "min1yr", "3 / 1, ≥2 months": "min1yr_2mo"}
SIZE_SPECS = {"baci_size": "BACI", "before_after_size": "before/after"}

def load_run(folder, run):
    for re_mode in ("auto", "slopes", "intercepts"):
        d = MODEL_DIR / folder / f"{run}__{re_mode}"
        if (d / "results.csv").exists():
            return pd.read_csv(d / "results.csv"), d
    return pd.DataFrame(), None

def model_rows(frame, specs):
    if frame.empty:
        return frame
    return frame[frame["spec"].isin(specs) & ~frame["series"].isin(rep.DIAGNOSTIC_SERIES)].copy()

main_runs = {run: load_run("stage3", run) for run in ("primary", "size_proxy")}
sens_runs = {label: {run: load_run(f"sensitivity_{v}", run) for run in ("primary", "size_proxy")}
             for label, v in VARIANTS.items()}
ready = [label for label, runs in sens_runs.items() if len(runs["primary"][0])]
print("Sensitivity runs found:", ready or "none")
if not ready:
    print("Sensitivity runs not done yet; see docs/eligibility_sensitivity_prespec.md (Procedure) for the commands.")


**Dams in each model, after every model filter** (perennial streams, suspect dams, exposure, both periods per station, covariates). These are the real sample sizes, smaller than the eligibility counts in `notebooks/qa/dam_funnel.ipynb`. Fewer than 10 = not fit; fewer than 15 = exploratory.

In [ ]:
if ready:
    counts = {"main (3 / 2)": model_rows(main_runs["primary"][0], DESIGN).set_index(["series", "spec"])["dams"]}
    for label in ready:
        counts[label] = model_rows(sens_runs[label]["primary"][0], DESIGN).set_index(["series", "spec"])["dams"]
    counts = pd.DataFrame(counts).fillna(0).astype(int)
    counts["gain (3 / 1)"] = counts.get("3 / 1", counts.iloc[:, 0]) - counts["main (3 / 2)"]
    counts = counts.rename(index=DESIGN, level="spec")
    wide = counts.unstack("spec").swaplevel(axis=1)
    order = [(d, c) for d in DESIGN.values() for c in counts.columns if (d, c) in wide.columns]
    display(wide[order])

**Estimates under each rule.** Effect at 5 km (÷ threshold), main and sensitivity side by side, colored by verdict. Rows that stay put = robust. The tables give the verdicts, labels and the change in the width of the 95% range (**width ratio** below 1 = more precise).

In [ ]:
if ready:
    est = ["effect_at_reference", "effect_ci_low", "effect_ci_high"]
    frames = [model_rows(main_runs["primary"][0], DESIGN).assign(rule="main (3 / 2)")]
    frames += [model_rows(sens_runs[l]["primary"][0], DESIGN).assign(rule=l) for l in ready]
    allr = pd.concat(frames, ignore_index=True)
    allr = allr[allr["error"].fillna("").eq("")]
    allr["effect_class"] = [rep.classify(e, lo, hi, rep.SESOI.get(s))
                            for e, lo, hi, s in zip(allr[est[0]], allr[est[1]], allr[est[2]], allr["series"])]
    allr = per_threshold(allr, est)
    order = ["main (3 / 2)"] + ready
    fig, axes = plt.subplots(1, 2, figsize=(14, 0.32 * allr["series"].nunique() * len(order) + 1.5))
    for ax, (spec, design) in zip(axes, DESIGN.items()):
        g = allr[allr["spec"] == spec].copy()
        g["k"] = g["rule"].map({r: i for i, r in enumerate(order)})
        g = g.sort_values(["series", "k"])
        g["label"] = g["series"] + " · " + g["rule"] + " (" + g["dams"].astype(int).astype(str) + ")"
        forest(ax, g, est[0], est[1], est[2], "label", "effect_class", title=f"{design}: effect at 5 km")
    fig.tight_layout(); plt.show()

    tables = {}
    for label in ready:
        for what, e, lo, hi, sesoi, mde in [("effect at 5 km", *est, rep.SESOI, "mde80_effect"),
                                             ("distance slope", "modifier_slope", "modifier_ci_low",
                                              "modifier_ci_high", None, "mde80_modifier")]:
            t = sc.compare(model_rows(main_runs["primary"][0], DESIGN), model_rows(sens_runs[label]["primary"][0], DESIGN),
                           ["series", "spec"], e, lo, hi, sesoi=sesoi, mde=mde)
            tables[(label, what)] = t.assign(rule=label, estimate=what)
    cmp = pd.concat(tables.values(), ignore_index=True)
    cmp["design"] = cmp["spec"].map(DESIGN)
    show = ["rule", "estimate", "design", "series", "dams_primary", "dams_sensitivity", "est_primary", "est_sensitivity",
            "verdict_primary", "verdict_sensitivity", "label", "negligible_sign_flip", "ci_width_ratio",
            "material_improvement"]
    for label in ready:
        print(f"\n{label} vs main:")
        display(cmp[cmp["rule"] == label][show[1:]].sort_values(["estimate", "design", "series"]).reset_index(drop=True))
    print("\nLabels, counted:")
    display(pd.crosstab([cmp["rule"], cmp["estimate"]], cmp["label"]))

**Reservoir size (exploratory).** The same comparison for the size models' slope: how the effect changes per SD of reservoir size.

In [ ]:
if ready and len(main_runs["size_proxy"][0]):
    for label in ready:
        if sens_runs[label]["size_proxy"][0].empty:
            print(f"{label}: no size_proxy run"); continue
        t = sc.compare(model_rows(main_runs["size_proxy"][0], SIZE_SPECS), model_rows(sens_runs[label]["size_proxy"][0], SIZE_SPECS),
                       ["series", "spec"], "modifier_slope", "modifier_ci_low", "modifier_ci_high")
        print(f"\n{label} vs main, size slope:")
        display(t.assign(design=t["spec"].map(SIZE_SPECS))[["design", "series", "dams_primary", "dams_sensitivity",
                 "est_primary", "est_sensitivity", "verdict_primary", "verdict_sensitivity", "label"]])

**Where do the extra dams materially help?** A measure counts as materially improved (fixed in advance) when its model becomes fittable (≥ 10 dams) or loses the exploratory label (≥ 15), its 95% range narrows by ≥ 20%, or its smallest detectable effect drops below the "too small to matter" threshold. Shown for 3 / 1, effect at 5 km.

In [ ]:
if ready:
    m = cmp[(cmp["rule"] == ready[0]) & (cmp["estimate"] == "effect at 5 km")]
    flags = [c for c in ["now_fittable", "no_longer_exploratory", "ci_narrower_20pct", "mde_now_below_sesoi"] if c in m]
    display(m[["design", "series", "dams_primary", "dams_sensitivity", "ci_width_ratio"] + flags + ["material_improvement"]]
            .sort_values(["design", "material_improvement"], ascending=[True, False]).reset_index(drop=True))

**Are the added dams different?** Dams in the 3 / 1 models but not in the main ones, compared with the main dams on the characteristics fixed in advance. **SMD** = standardized mean difference (difference in means ÷ pooled SD); **|SMD| ≥ 0.25 = systematic**. If a result changes *and* the added dams differ, the change may come from *who* was added, not from more data. The table gives means and medians; the SMD is computed from the means, so a skewed characteristic can show different medians with a small SMD.

In [ ]:
if ready:
    try:
        # Imported here so this cell runs on its own (it doesn't rely on the map cell).
        from etl.common.config import load_config
        from etl.common.warehouse import analysis_path, warehouse_dir
        WH = Path(os.environ.get("DAM_REMOVAL_WAREHOUSE_DIR") or warehouse_dir(load_config(CASE_STUDY)))
        cols = ["AR_ID", "station_id", "side"] + sorted({c for c, _, _ in sc.CHARACTERISTICS.values()})
        chars = sc.dam_characteristics(pd.read_parquet(analysis_path(WH, VARIANTS[ready[0]]), columns=cols))
    except Exception as exc:
        chars = None
        print(f"Characteristics need the warehouse table analysis_measurement_{VARIANTS[ready[0]]}.parquet: {exc}")
    if chars is not None:
        main_used = pd.read_csv(main_runs["primary"][1] / "dams_used.csv", dtype={"AR_ID": str})
        sens_used = pd.read_csv(sens_runs[ready[0]]["primary"][1] / "dams_used.csv", dtype={"AR_ID": str})
        for spec, design in DESIGN.items():
            p_ids = set(main_used.loc[main_used["spec"] == spec, "AR_ID"])
            added = set(sens_used.loc[sens_used["spec"] == spec, "AR_ID"]) - p_ids
            print(f"\n{design}: {len(p_ids)} dams in the main models, {len(added)} added under {ready[0]}")
            display(sc.compare_dams(chars, p_ids, added))

**Do the single-month pairs matter?** Each measure's label against the main results under 3 / 1 and under 3 / 1 with ≥ 2 months. Conclusions are unaffected by single-month pairs if the labels agree everywhere (fixed in advance); the table lists any that differ.

In [ ]:
if len(ready) == 2:
    a = cmp[cmp["rule"] == ready[0]].set_index(["estimate", "design", "series"])
    b = cmp[cmp["rule"] == ready[1]].set_index(["estimate", "design", "series"])
    both = a[["label", "verdict_sensitivity"]].join(b[["label", "verdict_sensitivity"]], lsuffix=" (3 / 1)", rsuffix=" (≥2 months)")
    differ = both[both["label (3 / 1)"] != both["label (≥2 months)"]]
    print(f"Labels differ for {len(differ)} of {len(both)} estimates")
    display(differ if len(differ) else pd.DataFrame({"note": ["none: single-month pairs don't change any conclusion"]}))
elif ready:
    print("Needs both variants (min1yr and min1yr_2mo).")

## 12. Sensitivity: BACI with controls known to be outside the former reservoir

The main BACI distance model drops upstream stations flagged as in the former reservoir, but counts stations whose status is **unknown** (dam height or slope missing) as controls. This refit of `baci_distance` keeps only controls **positively known** to be outside (pre-specified in `docs/control_sensitivity_prespec.md`, before any fit), on the main 3 / 2 data with the same random-effects rule. Compared like section 11: the effect at 5 km and the distance slope, with the pre-specified labels and the 10-dam rule. Then: how many BACI dams lose their controls, and whether the dams that remain differ from those that drop out.

In [ ]:
kc_run, kc_dir = load_run("sensitivity_known_controls", "known_controls")
if kc_run.empty:
    print("Not run yet: RUNS=known_controls sbatch data/slurm/fit_distance_models.slurm")
else:
    main_baci = model_rows(main_runs["primary"][0], {"baci_distance": "BACI"})
    kc_baci = model_rows(kc_run, {"baci_distance": "BACI"})
    kc_tables = []
    for what, e, lo, hi, sesoi in [("effect at 5 km", "effect_at_reference", "effect_ci_low", "effect_ci_high", rep.SESOI),
                                   ("distance slope", "modifier_slope", "modifier_ci_low", "modifier_ci_high", None)]:
        t = sc.compare(main_baci, kc_baci, ["series", "spec"], e, lo, hi, sesoi=sesoi)
        kc_tables.append(t.assign(estimate=what))
    kc_cmp = pd.concat(kc_tables, ignore_index=True)
    kc_cmp["dams_lost"] = kc_cmp["dams_primary"].fillna(0) - kc_cmp["dams_sensitivity"].fillna(0)
    print(f"BACI distance model, main vs controls known outside (fewer than {sc.MIN_DAMS} dams: shown, not compared):")
    display(kc_cmp[["estimate", "series", "dams_primary", "dams_sensitivity", "dams_lost", "est_primary", "est_sensitivity",
                    "verdict_primary", "verdict_sensitivity", "label", "negligible_sign_flip", "ci_width_ratio"]]
            .sort_values(["estimate", "series"]).reset_index(drop=True))
    print("Labels, counted:")
    display(pd.crosstab(kc_cmp["estimate"], kc_cmp["label"]))

In [ ]:
# Do the BACI dams that keep confirmed controls differ from those that lose them?
if not kc_run.empty and kc_dir is not None:
    try:
        from etl.common.config import load_config
        from etl.common.warehouse import analysis_path, warehouse_dir
        WH = Path(os.environ.get("DAM_REMOVAL_WAREHOUSE_DIR") or warehouse_dir(load_config(CASE_STUDY)))
        cols = ["AR_ID", "station_id", "side"] + sorted({c for c, _, _ in sc.CHARACTERISTICS.values()})
        main_chars = sc.dam_characteristics(pd.read_parquet(analysis_path(WH), columns=cols))
    except Exception as exc:
        main_chars = None
        print(f"Characteristics need the warehouse table analysis_measurement.parquet: {exc}")
    if main_chars is not None:
        used_main = pd.read_csv(main_runs["primary"][1] / "dams_used.csv", dtype={"AR_ID": str})
        used_kc = pd.read_csv(kc_dir / "dams_used.csv", dtype={"AR_ID": str})
        p_ids = set(used_main.loc[used_main["spec"] == "baci_distance", "AR_ID"])
        kept = set(used_kc.loc[used_kc["spec"] == "baci_distance", "AR_ID"]) & p_ids
        lost = p_ids - kept
        print(f"BACI dams in the main model: {len(p_ids)}; keep confirmed controls: {len(kept)}; lose them: {len(lost)}")
        print("Dams that lose their controls vs dams that keep them (SMD = lost minus kept):")
        display(sc.compare_dams(main_chars, kept, lost))

## 13. What could these models detect? (post-hoc design sensitivity)

**Post-hoc design sensitivity, not a pre-specified or prospective power analysis.** Added 2026-11-09, after the results were seen. It uses only the standard errors already in the result files (nothing is refit) and changes no estimate, verdict or threshold. It answers: *given how precise each model turned out to be, how large would a true effect have to be for it to be detected?*

- **MDE80** (minimum detectable effect): the true effect the model would detect (95% interval excluding 0) 80% of the time, = (t₀.₉₇₅ + t₀.₈₀) × SE. For log-scale measures the MDE is a log10 size, shown as the matching % rise / % fall (the same log size is a smaller % fall: +25% ↔ −20%).
- **Power at threshold:** the chance of detecting a true effect **exactly the size of the analyst-selected, provisional "too small to matter" threshold**. It is never computed at the observed estimate (that only restates the p-value). MDE and threshold are on the same scale (log10 for log-scale measures), so **MDE ÷ threshold < 1** ⇔ power at threshold > 80%.
- **Degrees of freedom: dams − 1**, the same as the models' 95% intervals (dams are the independent units). For a mixed model this is an approximation (statsmodels has no Satterthwaite / Kenward–Roger correction); it is likely conservative when the effect is estimated mostly within dams (random intercepts only) and may be optimistic when a few dams carry most of the data (fewer effective dams). The SEs themselves are the models' large-sample SEs.
- **Dams for MDE ≤ threshold** is a **rough projection, not a sample-size requirement**: it assumes the SE shrinks like 1 / √dams and that added dams resemble these (same variances, stations and sampling).
- **Reading it:** an *inconclusive* verdict with low power at threshold means the data were too thin to tell, not that nothing changed. A *detected* verdict doesn't need power to be believed, and low power is not a reason to discount it (though small, low-powered studies tend to overstate the size of what they detect).

Below: (a) the effect at 5 km for every primary model, (b) how detectability falls with distance, (c) the distance slopes.

In [ ]:
from scipy import stats
from analysis import distance_models as dm   # mde(), power(), dams_needed()

def is_log(series):
    """Log10 scale, as for the thresholds (stage4_report.sesoi_label)."""
    return pd.Series(series).str.split(" (", regex=False).str[0].isin(mm.SETTINGS["log_parameters"]).to_numpy()

def k80(dams):
    """MDE80 = k80 x SE with dams - 1 df (distance_models.mde), so SE = MDE80 / k80."""
    df = np.maximum(np.asarray(dams, float) - 1, 1)
    return stats.t.ppf(0.975, df) + stats.t.ppf(dm.POWER, df)

def plain_size(value, log_scale, unit=""):
    """A size on the model scale in plain units. Log10: the % rise / % fall of that size."""
    if pd.isna(value):
        return ""
    if log_scale:
        return f"+{(10 ** value - 1) * 100:.0f}% / −{(1 - 10 ** -value) * 100:.0f}%"
    return f"{value:.2g} {unit}".strip()

def needed_text(mde, threshold, dams):
    """Rough projection only (SE shrinking like 1 / sqrt(dams), new dams like these)."""
    if pd.isna(mde) or pd.isna(threshold):
        return ""
    if mde <= threshold:
        return "already ≤ threshold"
    n = dm.dams_needed(mde, threshold, dams)
    return "> 10,000" if np.isinf(n) else f"~{n:,.0f}"

def design_sensitivity(frame, mde=None, se=None):
    """Add threshold, MDE80 (plain units), MDE ÷ threshold, power at the threshold and the
    rough dams projection. Give the MDE80 column or the SE column (model units: log10 for
    log-scale series, the same scale as rep.SESOI)."""
    f = frame.copy()
    thr = f["series"].map(rep.SESOI).astype(float)
    dams = f["dams"].astype(float)
    se_v = f[se].astype(float) if se else f[mde].astype(float) / k80(dams)
    mde_v = f[mde].astype(float) if mde else se_v * k80(dams)
    log = is_log(f["series"])
    unit = f["unit"] if "unit" in f else pd.Series("", index=f.index)
    f["threshold"] = f["series"].map(rep.sesoi_label)
    f["MDE80"] = [plain_size(m, l, u) for m, l, u in zip(mde_v, log, unit.fillna(""))]
    f["MDE ÷ threshold"] = (mde_v / thr).round(2)
    f["power at threshold"] = np.round(dm.power(thr.to_numpy(), se_v.to_numpy(), dams.to_numpy()), 2)
    f["dams for MDE ≤ threshold (rough)"] = [needed_text(m, t, d) for m, t, d in zip(mde_v, thr, dams)]
    return f

# (a) Effect at 5 km, primary distance models (RE = auto). MDE80 comes from the result files.
ds = design_sensitivity(primary[primary["spec"].isin(DESIGN)], mde="mde80_effect")
ds["design"] = ds["spec"].map(DESIGN)
cols = ["design", "series", "dams", "sample_flag", "effect_class", "threshold", "MDE80", "MDE ÷ threshold",
        "power at threshold", "dams for MDE ≤ threshold (rough)"]
print("(a) Effect at 5 km: what each model could detect (power at the analyst-selected, provisional threshold):")
display(ds[[c for c in cols if c in ds]].sort_values(["design", "MDE ÷ threshold"]).reset_index(drop=True))
p = ds["power at threshold"]
print(f"{int((p >= 0.8).sum())} of {int(p.notna().sum())} models have ≥ 80% power at their threshold; "
      f"{int((p < 0.5).sum())} have < 50%.")

In [ ]:
# (b) MDE80 ÷ threshold at 1-30 km (effects_at.csv, primary run). Below 1 = a threshold-sized
# effect at that distance would be detected at least 80% of the time.
bd = by_distance[by_distance["primary"] & by_distance["spec"].isin(DESIGN) & by_distance["distance_km"].notna()
                 & by_distance["sesoi"].notna()].copy() if "mde80" in by_distance else pd.DataFrame()
if bd.empty:
    print("No effects_at rows with mde80 (rerun the report: rep.build).")
else:
    bd["ratio"] = bd["mde80"] / bd["sesoi"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
    for ax, (spec, design) in zip(axes, DESIGN.items()):
        for series, g in bd[bd["spec"] == spec].groupby("series"):
            g = g.sort_values("distance_km")
            ax.plot(g["distance_km"], g["ratio"], "o-", ms=3, lw=1.2, label=f"{series} ({int(g['dams'].iloc[0])})")
        ax.axhline(1, color="black", lw=0.8, ls="--")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xticks([1, 5, 10, 20, 30], ["1", "5", "10", "20", "30"])
        ticks = [t for t in (0.1, 0.25, 0.5, 1, 2, 4, 8, 16) if bd["ratio"].min() / 1.5 <= t <= bd["ratio"].max() * 1.5]
        ax.set_yticks(ticks, [f"{t:g}" for t in ticks]); ax.minorticks_off()
        ax.set_xlabel("km downstream"); ax.set_title(f"{design}: MDE80 ÷ threshold", fontsize=10)
        ax.legend(fontsize=7, frameon=False)
    axes[0].set_ylabel("MDE80 ÷ threshold")
    fig.suptitle("Below the dashed line: > 80% power to detect a threshold-sized effect at that distance", fontsize=9)
    fig.tight_layout(); plt.show()

In [ ]:
# (c) Distance slopes. A slope has no threshold of its own; as a yardstick, a slope of one
# threshold per unit of ln(1 + km) = the effect changing by one threshold between 5 and ~15 km.
sl = design_sensitivity(primary[primary["spec"].isin(DESIGN) & primary["mde80_modifier"].notna()],
                        mde="mde80_modifier")
sl["design"] = sl["spec"].map(DESIGN)
sl = sl.rename(columns={"MDE ÷ threshold": "slope MDE ÷ threshold",
                        "power at threshold": "power: one-threshold change 5→15 km"})
print("(c) Distance slopes:")
display(sl[["design", "series", "dams", "slope_class", "slope MDE ÷ threshold", "power: one-threshold change 5→15 km"]]
        .sort_values(["design", "slope MDE ÷ threshold"]).reset_index(drop=True))

**How to read section 13.** Rows near the top of (a) (MDE ÷ threshold < 1) are the measures where a threshold-sized change would very likely have been seen, so an *inconclusive* or *no meaningful change* verdict there is informative. Rows with MDE ÷ threshold well above 1 could only detect large changes; their *inconclusive* verdicts mean "too few dams or too noisy to tell". The figure shows where along the river each model could detect a threshold-sized effect: where a curve is above the dashed line, section 2's distance curves can't tell a threshold-sized effect from none at that distance. These are the same SEs and the same dams − 1 df as the 95% intervals above, so this section adds a reading, not a new result.

**Notes:** distance is measured along the river. The drainage-area ratio is an estimate of dilution, not a flow measurement. The reservoir-size index comes from dam height, slope and river size; it is not a measured volume or sediment amount. Twelve dams suspected of being placed on the wrong stream are excluded (`config/suspect_dam_matches.csv`). Full tables: `stage4/report/`. Section 11's sensitivity runs are in `sensitivity_min1yr/` and `sensitivity_min1yr_2mo/`. How the effect changes over the years after removal, and which dam characteristics go with bigger or smaller effects: `time_and_modifier_results.ipynb`.